# 🔍 Week 6: Debugging ML Models & Reasoning Skills

## Overview
This is where you differentiate yourself from other candidates. Knowing WHY things fail and HOW to fix them is critical.

## 🎯 Learning Objectives
1. Diagnose bias vs variance issues
2. Understand precision vs recall trade-offs
3. Debug common ML problems
4. Reason about model behavior

## 💡 Interview Insight
"Your model has 70% accuracy. What would you try to improve it?" - Answer this systematically!

---

In [ ]:
# ============================================================
# IMPORTS AND SETUP
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, roc_auc_score,
                             mean_squared_error, r2_score)
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.style.use('seaborn-v0_8-whitegrid')

print("Setup complete!")

---
## 1. Bias vs Variance Trade-off

### The Most Common Interview Question!

**Bias:** Error from oversimplified model (underfitting)
**Variance:** Error from over-sensitive model (overfitting)

In [ ]:
# ============================================================
# VISUALIZE BIAS VS VARIANCE
# ============================================================

# Generate non-linear data
np.random.seed(RANDOM_STATE)
X = np.linspace(0, 10, 100).reshape(-1, 1)
y_true = np.sin(X).ravel()
y = y_true + np.random.normal(0, 0.2, 100)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Three models with different complexity
models = {
    'High Bias (Linear)': LinearRegression(),
    'Balanced (Degree 4)': None,  # Polynomial
    'High Variance (Degree 15)': None  # Polynomial
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X_plot = np.linspace(0, 10, 200).reshape(-1, 1)

# Model 1: High Bias (Linear)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

axes[0].scatter(X_train, y_train, c='blue', alpha=0.5, label='Train')
axes[0].scatter(X_test, y_test, c='red', alpha=0.5, label='Test')
axes[0].plot(X_plot, model.predict(X_plot), 'g-', lw=2, label='Prediction')
axes[0].set_title(f'High Bias (Underfitting)\nTrain MSE: {mean_squared_error(y_train, y_pred_train):.3f}\nTest MSE: {mean_squared_error(y_test, y_pred_test):.3f}')
axes[0].legend()

# Model 2: Balanced (Degree 4)
poly = PolynomialFeatures(degree=4)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
model = LinearRegression()
model.fit(X_train_poly, y_train)
y_pred_train = model.predict(X_train_poly)
y_pred_test = model.predict(X_test_poly)

axes[1].scatter(X_train, y_train, c='blue', alpha=0.5, label='Train')
axes[1].scatter(X_test, y_test, c='red', alpha=0.5, label='Test')
axes[1].plot(X_plot, model.predict(poly.transform(X_plot)), 'g-', lw=2, label='Prediction')
axes[1].set_title(f'Balanced (Good Fit)\nTrain MSE: {mean_squared_error(y_train, y_pred_train):.3f}\nTest MSE: {mean_squared_error(y_test, y_pred_test):.3f}')
axes[1].legend()

# Model 3: High Variance (Degree 15)
poly = PolynomialFeatures(degree=15)
X_train_poly = poly.fit_transform(X_train)
X_test_poly = poly.transform(X_test)
model = LinearRegression()
model.fit(X_train_poly, y_train)
y_pred_train = model.predict(X_train_poly)
y_pred_test = model.predict(X_test_poly)

axes[2].scatter(X_train, y_train, c='blue', alpha=0.5, label='Train')
axes[2].scatter(X_test, y_test, c='red', alpha=0.5, label='Test')
y_plot = model.predict(poly.transform(X_plot))
y_plot = np.clip(y_plot, -2, 2)  # Clip for visualization
axes[2].plot(X_plot, y_plot, 'g-', lw=2, label='Prediction')
axes[2].set_title(f'High Variance (Overfitting)\nTrain MSE: {mean_squared_error(y_train, y_pred_train):.3f}\nTest MSE: {mean_squared_error(y_test, y_pred_test):.3f}')
axes[2].legend()
axes[2].set_ylim(-2, 2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DIAGNOSTIC: LEARNING CURVES
# ============================================================

print("📋 How to Read Learning Curves:")
print("="*60)
print("""
HIGH BIAS (Underfitting):
  - Both train and validation scores are LOW
  - They converge quickly but to a poor value
  - Adding more data won't help much
  → Solution: Increase model complexity

HIGH VARIANCE (Overfitting):
  - Train score is HIGH, validation score is LOW
  - Large gap between curves
  - Gap decreases with more data
  → Solution: Regularization, more data, simpler model
  
GOOD FIT:
  - Both scores are high
  - Small gap between them
  - Curves converge
""")

In [ ]:
# ============================================================
# PLOT LEARNING CURVES
# ============================================================

from sklearn.datasets import make_classification

# Create dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

models = [
    ('High Bias', LogisticRegression(C=0.001, max_iter=1000)),
    ('Balanced', LogisticRegression(C=1.0, max_iter=1000)),
    ('High Variance', DecisionTreeClassifier(max_depth=None))
]

for ax, (name, model) in zip(axes, models):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, n_jobs=-1,
        train_sizes=np.linspace(0.1, 1.0, 10),
        scoring='accuracy'
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std = train_scores.std(axis=1)
    val_mean = val_scores.mean(axis=1)
    val_std = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='blue')
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='orange')
    ax.plot(train_sizes, train_mean, 'o-', color='blue', label='Training score')
    ax.plot(train_sizes, val_mean, 'o-', color='orange', label='Validation score')
    ax.set_xlabel('Training Size')
    ax.set_ylabel('Score')
    ax.set_title(name)
    ax.legend(loc='lower right')
    ax.set_ylim(0.5, 1.05)

plt.tight_layout()
plt.show()

---
## 2. Precision vs Recall Trade-off

### Critical for imbalanced classification!

In [ ]:
# ============================================================
# PRECISION VS RECALL EXPLAINED
# ============================================================

print("📋 Precision vs Recall:")
print("="*60)
print("""
PRECISION: Of all predicted positives, how many are correct?
  - Formula: TP / (TP + FP)
  - High precision = Few false positives
  - Important when: Cost of false positive is high
  - Example: Spam detection (don't want good emails in spam)

RECALL: Of all actual positives, how many did we catch?
  - Formula: TP / (TP + FN)  
  - High recall = Few false negatives
  - Important when: Cost of missing positives is high
  - Example: Disease detection (don't want to miss sick patients)

F1 SCORE: Harmonic mean of precision and recall
  - Formula: 2 * (P * R) / (P + R)
  - Use when: You need balance between precision and recall
""")

In [ ]:
# ============================================================
# VISUALIZE PRECISION-RECALL TRADE-OFF
# ============================================================

from sklearn.metrics import precision_recall_curve

# Create imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           weights=[0.9, 0.1], random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate precision-recall curve
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Precision-Recall Curve
axes[0].plot(recalls, precisions, 'b-', lw=2)
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve')
axes[0].fill_between(recalls, precisions, alpha=0.2)
axes[0].set_xlim([0, 1])
axes[0].set_ylim([0, 1])

# Plot 2: Precision and Recall vs Threshold
axes[1].plot(thresholds, precisions[:-1], 'b-', label='Precision', lw=2)
axes[1].plot(thresholds, recalls[:-1], 'r-', label='Recall', lw=2)
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('Score')
axes[1].set_title('Precision/Recall vs Threshold')
axes[1].legend()
axes[1].axvline(x=0.5, color='gray', linestyle='--', label='Default threshold')

plt.tight_layout()
plt.show()

print("📌 Key Insight:")
print("  - Lower threshold → Higher recall, Lower precision")
print("  - Higher threshold → Lower recall, Higher precision")

In [ ]:
# ============================================================
# SCENARIO-BASED THRESHOLD SELECTION
# ============================================================

print("📋 Choosing the Right Metric - Interview Scenarios:")
print("="*60)

scenarios = [
    {
        'scenario': 'Cancer Detection',
        'priority': 'HIGH RECALL',
        'reason': 'Missing a cancer case could be fatal',
        'accept': 'Some healthy patients flagged for extra tests'
    },
    {
        'scenario': 'Spam Email Filter',
        'priority': 'HIGH PRECISION',
        'reason': 'Important emails going to spam is costly',
        'accept': 'Some spam getting through'
    },
    {
        'scenario': 'Fraud Detection',
        'priority': 'BALANCED (F1)',
        'reason': 'Both missing fraud and blocking legit transactions are costly',
        'accept': 'Optimize for business-specific cost'
    },
    {
        'scenario': 'Content Moderation',
        'priority': 'HIGH RECALL',
        'reason': 'Missing harmful content can cause damage',
        'accept': 'Some false flags can be reviewed by humans'
    }
]

for s in scenarios:
    print(f"\n🎯 {s['scenario']}")
    print(f"   Priority: {s['priority']}")
    print(f"   Reason: {s['reason']}")
    print(f"   Trade-off: {s['accept']}")

---
## 3. Debugging Checklist: "What to Try When Results Suck"

In [ ]:
# ============================================================
# THE DEBUGGING FRAMEWORK
# ============================================================

print("📋 ML Debugging Framework:")
print("="*60)
print("""
STEP 1: DIAGNOSE THE PROBLEM
============================
□ Check training vs validation error
  - Both high → High Bias (underfitting)
  - Train low, val high → High Variance (overfitting)
  
□ Examine learning curves
□ Check for data issues (leakage, imbalance)


STEP 2: FIX HIGH BIAS (UNDERFITTING)
=====================================
□ Add more features / feature engineering
□ Use more complex model
□ Reduce regularization
□ Train longer (for neural networks)
□ Decrease dropout (for neural networks)


STEP 3: FIX HIGH VARIANCE (OVERFITTING)
========================================
□ Get more training data
□ Reduce model complexity
□ Increase regularization (L1, L2)
□ Add dropout (for neural networks)
□ Use early stopping
□ Feature selection (remove noise)
□ Use ensemble methods


STEP 4: DATA-CENTRIC IMPROVEMENTS
==================================
□ Fix data quality issues
□ Handle class imbalance
□ Better train/test split (stratified, time-based)
□ Data augmentation
□ Clean noisy labels
""")

In [ ]:
# ============================================================
# COMMON ISSUES AND SOLUTIONS
# ============================================================

print("🔧 Common ML Problems & Solutions:")
print("="*60)

problems = [
    {
        'problem': 'Model accuracy is 95% but business says it\'s terrible',
        'likely_cause': 'Class imbalance - 95% of data is one class',
        'solution': 'Use F1, precision/recall, or AUC instead of accuracy'
    },
    {
        'problem': 'Training accuracy is 99%, test accuracy is 60%',
        'likely_cause': 'Overfitting',
        'solution': 'Regularization, more data, simpler model'
    },
    {
        'problem': 'Model performs great in dev, fails in production',
        'likely_cause': 'Data distribution shift or data leakage',
        'solution': 'Check for leakage, ensure prod data matches train data'
    },
    {
        'problem': 'Adding features made model worse',
        'likely_cause': 'Curse of dimensionality or noise features',
        'solution': 'Feature selection, regularization'
    },
    {
        'problem': 'Model predictions are always the majority class',
        'likely_cause': 'Severe class imbalance',
        'solution': 'Class weights, resampling, threshold adjustment'
    },
    {
        'problem': 'Cross-validation scores vary wildly',
        'likely_cause': 'Small dataset or high variance model',
        'solution': 'More data, simpler model, stratified CV'
    }
]

for i, p in enumerate(problems, 1):
    print(f"\n{i}. PROBLEM: {p['problem']}")
    print(f"   LIKELY CAUSE: {p['likely_cause']}")
    print(f"   SOLUTION: {p['solution']}")

---
## 4. Practical Debugging Example

In [ ]:
# ============================================================
# DEBUGGING CASE STUDY: IMBALANCED CLASSIFICATION
# ============================================================

# Create heavily imbalanced dataset
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           weights=[0.95, 0.05], random_state=RANDOM_STATE)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print("Dataset Info:")
print(f"  Training: {len(y_train)} samples")
print(f"  Class distribution: {np.bincount(y_train)}")
print(f"  Minority class: {np.bincount(y_train)[1] / len(y_train) * 100:.1f}%")

# Train naive model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\n--- NAIVE MODEL ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.3f}")
print(f"Recall: {recall_score(y_test, y_pred, zero_division=0):.3f}")
print(f"F1: {f1_score(y_test, y_pred, zero_division=0):.3f}")

print("\n⚠️ Problem: High accuracy but low recall!")
print("   Model is just predicting the majority class.")

In [ ]:
# ============================================================
# SOLUTION 1: CLASS WEIGHTS
# ============================================================

model_balanced = LogisticRegression(class_weight='balanced', max_iter=1000)
model_balanced.fit(X_train, y_train)
y_pred_balanced = model_balanced.predict(X_test)

print("--- WITH CLASS WEIGHTS ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred_balanced):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_balanced, zero_division=0):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_balanced, zero_division=0):.3f}")
print(f"F1: {f1_score(y_test, y_pred_balanced, zero_division=0):.3f}")

print("\n✅ Improvement: Much better recall!")

In [ ]:
# ============================================================
# SOLUTION 2: THRESHOLD ADJUSTMENT
# ============================================================

y_prob = model.predict_proba(X_test)[:, 1]

# Try different thresholds
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5]

print("--- THRESHOLD TUNING ---")
print(f"{'Threshold':>10} {'Precision':>12} {'Recall':>10} {'F1':>10}")
print("-" * 45)

for thresh in thresholds:
    y_pred_thresh = (y_prob >= thresh).astype(int)
    precision = precision_score(y_test, y_pred_thresh, zero_division=0)
    recall = recall_score(y_test, y_pred_thresh, zero_division=0)
    f1 = f1_score(y_test, y_pred_thresh, zero_division=0)
    print(f"{thresh:>10.1f} {precision:>12.3f} {recall:>10.3f} {f1:>10.3f}")

print("\n✅ Lower threshold → Higher recall")

---
## 5. Interview Questions & Answers

In [ ]:
# ============================================================
# COMMON INTERVIEW QUESTIONS
# ============================================================

print("📋 Top Interview Questions & Answers:")
print("="*60)

qa = [
    {
        'q': 'Your model has 70% accuracy. What would you try?',
        'a': '''First diagnose: Is it underfitting or overfitting?
   - Compare train vs validation accuracy
   - Look at learning curves
   Then apply appropriate fixes:
   - Underfitting: More complex model, more features
   - Overfitting: Regularization, more data, simpler model'''
    },
    {
        'q': 'When would you use precision over recall?',
        'a': '''Use precision when false positives are costly:
   - Spam detection (don't want good emails in spam)
   - Recommendation systems (don't annoy users)
   Use recall when false negatives are costly:
   - Disease detection (don't miss sick patients)
   - Fraud detection (must catch fraud)'''
    },
    {
        'q': 'Model works in development but fails in production. Why?',
        'a': '''Common causes:
   1. Data leakage in training
   2. Distribution shift (prod data differs from train)
   3. Missing feature processing in production
   4. Different data quality/preprocessing
   Debug: Compare feature distributions and check pipeline'''
    },
    {
        'q': 'How do you handle class imbalance?',
        'a': '''Options:
   1. Class weights in model
   2. Oversampling minority (SMOTE)
   3. Undersampling majority
   4. Adjust prediction threshold
   5. Use appropriate metrics (F1, AUC, not accuracy)'''
    },
    {
        'q': 'Adding more features made my model worse. Why?',
        'a': '''Possible reasons:
   1. Noisy/irrelevant features adding variance
   2. Curse of dimensionality
   3. Multicollinearity
   Solutions:
   - Feature selection
   - Regularization (L1/Lasso for sparsity)
   - Dimensionality reduction (PCA)'''
    }
]

for i, item in enumerate(qa, 1):
    print(f"\n{i}. Q: {item['q']}")
    print(f"   A: {item['a']}")

---
## ✅ Week 6 Checklist

- [x] Understand bias vs variance trade-off
- [x] Read and interpret learning curves
- [x] Know precision vs recall trade-offs
- [x] Have systematic debugging approach
- [x] Handle class imbalance
- [x] Know common problems and solutions

---

**Next: Week 7 - Coding & Communication** 🚀